# 10 — Spatial and distribution plotting routines

This notebook checks the spatial colormap and parameter-distribution outputs of `FittingResultPlotter` on a parameter field with a designed spatial structure. Because the inputs are controlled, the rendered maps can be verified against the structure that was built in.

Modules exercised:

- `pipelines.param_pipeline.plots.FittingResultPlotter` (real; individual plot methods called)
- `pipelines.param_pipeline.metrics.FittingMetricsCalculator` (real, imported and run)
- `tools.reporting.plotting.PlotBase` (style and clim helpers)

Synthetic inputs, fixed seed, CPU only.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt

SEED = 0
rng  = np.random.default_rng(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    torch = None

plt.rcParams.update({
    "figure.dpi"     : 110,
    "savefig.dpi"    : 110,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.30,
    "grid.linewidth" : 0.4,
})

print("Repo root :", REPO_ROOT)
print("NumPy     :", np.__version__)


In [ ]:
def gaussian_mixture_profile(height_axis, components, noise_std=0.0, rng=None):
    profile = np.zeros_like(height_axis, dtype=np.float64)
    for amp, mu, sigma in components:
        profile += amp * np.exp(-((height_axis - mu) ** 2) / (2.0 * sigma * sigma))
    if noise_std > 0.0:
        draw    = rng.normal(0.0, noise_std, size=height_axis.shape) if rng is not None else np.random.normal(0.0, noise_std, size=height_axis.shape)
        profile = profile + draw
    return profile.astype(np.float32)


def pack_parameters(components_per_pixel, k_max, shape):
    Az, R  = shape
    params = np.zeros((3 * k_max, Az, R), dtype=np.float32)
    for (az, rg), comps in components_per_pixel.items():
        for k, (amp, mu, sigma) in enumerate(comps[:k_max]):
            params[3 * k    , az, rg] = amp
            params[3 * k + 1, az, rg] = mu
            params[3 * k + 2, az, rg] = sigma
    return params


In [ ]:
import tempfile
from pathlib import Path

import matplotlib.image as mpimg

from tools.data.gaussians import GaussianMixture
from pipelines.param_pipeline.metrics import FittingMetricsCalculator
from pipelines.param_pipeline.plots   import FittingResultPlotter
from tools.monitoring.logger import Logger

logger = Logger(log_dir='', name='nb10_plots')
print('imported')

## Parameter field with a centroid gradient

Component 0 has a centroid that increases linearly along range, so its mu map should show a smooth left-to-right gradient. Component 1 has a constant centroid and an amplitude that increases along azimuth, so its amplitude map should show a top-to-bottom gradient. These controlled gradients make the spatial maps checkable by eye.

In [ ]:
K            = 2
Az, R        = 20, 40
H            = 80
height_range = (0.0, 40.0)
height_axis  = np.linspace(*height_range, H).astype(np.float32)

rr = np.linspace(0.0, 1.0, R)[None, :] * np.ones((Az, 1))
aa = np.linspace(0.0, 1.0, Az)[:, None] * np.ones((1, R))

params = np.zeros((3 * K, Az, R), dtype=np.float32)
params[0] = 1.0
params[1] = 8.0 + 20.0 * rr
params[2] = 2.5
params[3] = 0.3 + 0.6 * aa
params[4] = 32.0
params[5] = 3.0

tomogram = np.zeros((H, Az, R), dtype=np.float32)
for j, h in enumerate(height_axis):
    tomogram[j] = GaussianMixture.evaluate_slice(params, float(h), K)
tomogram += rng.normal(0.0, 0.005, tomogram.shape).astype(np.float32)

tmp_dir  = Path(tempfile.mkdtemp())
tom_path = tmp_dir / 'gradient_tomogram.npy'
np.save(tom_path, tomogram)
metadata = {'height_range': list(height_range)}
print('gradient parameter field ready')

## Run metrics and the spatial / distribution plot methods

We compute the metrics dictionary, apply the plotter style, set up its output directories, and call the individual spatial-map, R-squared-map, and parameter-distribution methods directly so each routine is exercised in isolation.

In [ ]:
calc    = FittingMetricsCalculator(n_gaussians=K, logger=logger)
metrics = calc.run(params, metadata, tom_path)

plotter = FittingResultPlotter(output_directory=tmp_dir, n_gaussians=K, logger=logger)
plotter._apply_style()
dirs = plotter._setup_output_dirs()

mu_keys   = [f'mu_{k}'  for k in range(K)]
mu_titles = [f'mu component {k + 1} [m]' for k in range(K)]
mu_path   = plotter._plot_spatial_maps(metrics, mu_keys, mu_titles, 'Centroid height maps', 'mu [m]', dirs['colormaps'] / 'mu_maps.png', cmap='RdYlGn')

amp_keys   = [f'amp_{k}' for k in range(K)]
amp_titles = [f'amplitude component {k + 1}' for k in range(K)]
amp_path   = plotter._plot_spatial_maps(metrics, amp_keys, amp_titles, 'Amplitude maps', 'A', dirs['colormaps'] / 'amp_maps.png', cmap='plasma')

r2_path   = plotter._plot_r2_spatial_map(metrics['r2_map'], dirs['colormaps'] / 'r2_map.png')
dist_paths = plotter._plot_parameter_distributions(params, dirs['distributions'])

print('mu map   :', mu_path.name)
print('amp map  :', amp_path.name)
print('r2 map   :', r2_path.name)
print('dist keys:', sorted(dist_paths))

In [ ]:
panels = [('centroid maps', mu_path), ('amplitude maps', amp_path), ('parameter distributions (mu)', dist_paths.get('mu_dist'))]
panels = [(t, p) for t, p in panels if p is not None]
fig, axes = plt.subplots(len(panels), 1, figsize=(12, 4.5 * len(panels)))
if len(panels) == 1:
    axes = [axes]
for ax, (title, path) in zip(axes, panels):
    ax.imshow(mpimg.imread(path))
    ax.set_axis_off()
    ax.set_title(title)
fig.tight_layout()
plt.show()

## Expected outcome

The component-0 centroid map should display a smooth left-to-right gradient (8 m rising to 28 m), and the component-1 amplitude map should display a top-to-bottom gradient. The R-squared map should be near 1 everywhere since the parameters generated the tomogram exactly. The distribution panel should summarise the parameter ranges that were built in, confirming the spatial and distribution plotting routines render controlled inputs faithfully.